# Notebook 1 — ANN with Learned Embeddings (MovieLens)

**Goal**: Train a neural network that uses `nn.Embedding` layers for categorical features,
then save those learned embedding weights to disk for reuse in classical models.

| | |
|---|---|
| **Categorical → embeddings** | `genres` (mean-pooled), `original_language` |
| **Numerical → MLP** | `imdbRating`, `imdbVotes`, `tmdbRating`, `tmdbVotes`, `popularity`, `budget`, `runtime`, `revenue`, `tag_count`, `avg_relevance`, `max_relevance` |
| **Ignored** | `userId`, `movieId` |
| **Target** | `rating` |

**Files saved after training**
```
embeddings/
  genre_embeddings.npy    # weight matrix shape: (n_genres, emb_dim)
  lang_embeddings.npy     # weight matrix shape: (n_langs,  emb_dim)
  genre_encoder.json      # {"Action": 0, "Drama": 1, ...}
  lang_encoder.json       # {"en": 0, "fr": 1, ...}
```


## 1. Setup & Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute        import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device   : {DEVICE}")

# ── Output folder for embeddings ──
EMB_DIR = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\embeddings"
os.makedirs(EMB_DIR, exist_ok=True)
print(f"Embeddings will be saved to: {EMB_DIR}")


## 2. Load Pre-split Data

Data is already merged, cleaned and split. We load directly — no joining needed.


In [ ]:
DATA_DIR = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\data"

df_train = pd.read_csv(DATA_DIR + '\\train.csv')
df_val   = pd.read_csv(DATA_DIR + '\\val.csv')
df_test  = pd.read_csv(DATA_DIR + '\\test.csv')

print(f"Train : {df_train.shape}")
print(f"Val   : {df_val.shape}")
print(f"Test  : {df_test.shape}")
print(f"\nColumns: {list(df_train.columns)}")


## 3. Define Feature Columns

We explicitly declare which columns play which role.
Anything not listed here is ignored (userId, movieId, date cols, etc.).


In [ ]:
TARGET      = 'rating'
CAT_GENRE   = 'genres'            # pipe-separated e.g. "Action|Drama|Thriller"
CAT_LANG    = 'original_language' # e.g. "en", "fr", "ko"

NUM_COLS = [
    'imdbRating', 'imdbVotes',
    'tmdbRating', 'tmdbVotes',
    'popularity', 'budget', 'runtime', 'revenue',
    'tag_count',  'avg_relevance', 'max_relevance'
]

# Quick check — confirm all columns exist
for col in NUM_COLS + [CAT_GENRE, CAT_LANG, TARGET]:
    assert col in df_train.columns, f"Missing column: {col}"

print("All expected columns confirmed present.")


## 4. Numerical Preprocessing

Steps applied **in order**, fit only on train:
1. **Median imputation** — missing values replaced with training-set median
2. **log1p** — compresses heavy right-skewed distributions (votes, revenue, budget)
3. **StandardScaler** — zero mean, unit variance (stabilises ANN gradient updates)


In [ ]:
# ── Step 1: Median imputation ──
# Fill tag features with 0 first (missing = user left no tag, not truly unknown)
for df_ in [df_train, df_val, df_test]:
    df_[['tag_count', 'avg_relevance', 'max_relevance']] =         df_[['tag_count', 'avg_relevance', 'max_relevance']].fillna(0)

imputer = SimpleImputer(strategy='median')
df_train[NUM_COLS] = imputer.fit_transform(df_train[NUM_COLS])  # fit ONLY on train
df_val[NUM_COLS]   = imputer.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = imputer.transform(df_test[NUM_COLS])

# ── Step 2: log1p transform ──
# clip to 0 first so log1p never sees negatives (budget/revenue can have -9999 sentinel)
for df_ in [df_train, df_val, df_test]:
    df_[NUM_COLS] = np.log1p(df_[NUM_COLS].clip(lower=0))

# ── Step 3: StandardScaler ──
scaler = StandardScaler()
df_train[NUM_COLS] = scaler.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = scaler.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = scaler.transform(df_test[NUM_COLS])

print("Numerical preprocessing done.")
print(df_train[NUM_COLS].describe().round(3))


## 5. Categorical Label Encoding

We need integer indices for `nn.Embedding` lookup tables.

**Language**: straightforward — one token per row → one integer index.

**Genres**: pipe-separated multi-label (e.g. `"Action|Drama"`).
- We first collect every unique individual genre token across the whole training set.
- At runtime each row's genres are split, each token is looked up, and their embedding
  vectors are **mean-pooled** into a single fixed-size vector.
- Unknown genres seen in val/test (none expected, but handled) map to index 0 (`<UNK>`).


In [ ]:
# ── Language encoder ──
# Collect all unique languages from train; unseen ones in val/test → '<UNK>'
lang_vocab = ['<UNK>'] + sorted(df_train[CAT_LANG].fillna('<UNK>').unique().tolist())
lang2idx   = {lang: i for i, lang in enumerate(lang_vocab)}

def encode_lang(series):
    """Map a language string to its integer index. Unknowns → 0."""
    return series.fillna('<UNK>').map(lambda x: lang2idx.get(x, 0)).values

train_lang = encode_lang(df_train[CAT_LANG])
val_lang   = encode_lang(df_val[CAT_LANG])
test_lang  = encode_lang(df_test[CAT_LANG])

print(f"Language vocab size : {len(lang2idx)}")
print(f"Sample lang indices : {train_lang[:5]}")


In [ ]:
# ── Genre encoder ──
# Collect all individual genre tokens from train (split pipe-separated strings)
all_genre_tokens = set()
for g in df_train[CAT_GENRE].dropna():
    all_genre_tokens.update(g.split('|'))

genre_vocab = ['<UNK>'] + sorted(all_genre_tokens)
genre2idx   = {g: i for i, g in enumerate(genre_vocab)}

def encode_genres(series, max_genres=10):
    """
    Convert a pipe-separated genre string into a padded integer sequence.
    Returns a 2D numpy array of shape (n_rows, max_genres).
    Rows with fewer than max_genres tokens are right-padded with -1
    (used later to mask out padding before mean-pooling).
    """
    result = []
    for g_str in series.fillna('<UNK>'):
        tokens  = g_str.split('|')[:max_genres]
        indices = [genre2idx.get(t, 0) for t in tokens]
        # Pad to max_genres with -1 (padding marker, not a real index)
        indices += [-1] * (max_genres - len(indices))
        result.append(indices)
    return np.array(result, dtype=np.int64)

MAX_GENRES  = 10  # no movie has more than ~8 genres in practice
train_genres = encode_genres(df_train[CAT_GENRE], MAX_GENRES)
val_genres   = encode_genres(df_val[CAT_GENRE],   MAX_GENRES)
test_genres  = encode_genres(df_test[CAT_GENRE],  MAX_GENRES)

print(f"Genre vocab size    : {len(genre2idx)}")
print(f"Genre matrix shape  : {train_genres.shape}")
print(f"Sample (first row)  : {train_genres[0]}  → {df_train[CAT_GENRE].iloc[0]}")


In [ ]:
# ── Save encoders to disk (needed by Notebook 2 to do the same lookups) ──
with open(os.path.join(EMB_DIR, 'genre_encoder.json'), 'w') as f:
    json.dump(genre2idx, f)
with open(os.path.join(EMB_DIR, 'lang_encoder.json'), 'w') as f:
    json.dump(lang2idx, f)

print("Encoders saved.")


## 6. PyTorch Dataset & DataLoader

Each sample returns a tuple of:
- `num_features` — float32 tensor of shape `(11,)`
- `genre_indices` — int64 tensor of shape `(MAX_GENRES,)` with –1 padding
- `lang_index`   — int64 scalar
- `label`        — float32 scalar (the rating)


In [ ]:
class MovieDataset(Dataset):
    def __init__(self, num_arr, genre_arr, lang_arr, labels):
        self.num    = torch.tensor(num_arr,   dtype=torch.float32)
        self.genres = torch.tensor(genre_arr, dtype=torch.long)
        self.lang   = torch.tensor(lang_arr,  dtype=torch.long)
        self.labels = torch.tensor(labels,    dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.num[idx], self.genres[idx], self.lang[idx], self.labels[idx]


def make_loader(df, genre_arr, lang_arr, shuffle):
    ds = MovieDataset(
        num_arr   = df[NUM_COLS].values.astype(np.float32),
        genre_arr = genre_arr,
        lang_arr  = lang_arr,
        labels    = df[TARGET].values.astype(np.float32)
    )
    return DataLoader(ds, batch_size=512, shuffle=shuffle, num_workers=0)

train_loader = make_loader(df_train, train_genres, train_lang, shuffle=True)
val_loader   = make_loader(df_val,   val_genres,   val_lang,   shuffle=False)
test_loader  = make_loader(df_test,  test_genres,  test_lang,  shuffle=False)

# Sanity check — inspect one batch
num_b, genre_b, lang_b, label_b = next(iter(train_loader))
print(f"num shape   : {num_b.shape}")
print(f"genre shape : {genre_b.shape}")
print(f"lang shape  : {lang_b.shape}")
print(f"label shape : {label_b.shape}")


## 7. EmbeddingANN Architecture

```
genres  →  nn.Embedding(n_genres, 8)  →  mean-pool (ignore -1 padding)  ──┐
lang    →  nn.Embedding(n_langs,  4)  →  direct lookup                   ──┤
num     →  (already scaled)                                               ──┤
                                                                            ↓
                                                    concat → [8 + 4 + 11 = 23]
                                                    Linear(23 → 64) → ReLU
                                                    Linear(64 → 32) → ReLU
                                                    Linear(32 →  1)  = predicted rating
```

**Embedding dimensions** — rule of thumb: `min(50, (vocab_size // 2) + 1)`.
With ~20 genres we use 8; with ~50 languages we use 4.
Small dims are intentional — we want compressed, generalisable representations.


In [ ]:
class EmbeddingANN(nn.Module):
    def __init__(self, n_genres, n_langs, n_num,
                 genre_emb_dim=8, lang_emb_dim=4):
        super().__init__()

        # ── Embedding tables ──
        # padding_idx=0 means the <UNK> token always produces a zero vector
        # (keeps unknowns neutral rather than training a noisy catch-all vector)
        self.genre_emb = nn.Embedding(n_genres, genre_emb_dim, padding_idx=0)
        self.lang_emb  = nn.Embedding(n_langs,  lang_emb_dim,  padding_idx=0)

        # ── MLP head ──
        mlp_input_dim = genre_emb_dim + lang_emb_dim + n_num
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, num, genre_idx, lang_idx):
        # ── Genre: embed each token then mean-pool, ignoring -1 padding ──
        # genre_idx shape: (batch, MAX_GENRES)
        # mask out padding positions (-1) before pooling
        mask = (genre_idx >= 0)                         # (batch, MAX_GENRES) bool
        safe_idx = genre_idx.clamp(min=0)               # replace -1 with 0 so embedding doesn't crash
        genre_vecs = self.genre_emb(safe_idx)           # (batch, MAX_GENRES, emb_dim)
        # zero out the padding positions
        genre_vecs = genre_vecs * mask.unsqueeze(-1).float()
        # mean over non-padding tokens; clamp denom to avoid div-by-zero
        genre_vec  = genre_vecs.sum(dim=1) / mask.sum(dim=1).clamp(min=1).unsqueeze(-1)
        # genre_vec shape: (batch, genre_emb_dim)

        # ── Language: single lookup ──
        lang_vec = self.lang_emb(lang_idx)              # (batch, lang_emb_dim)

        # ── Concatenate all inputs ──
        x = torch.cat([genre_vec, lang_vec, num], dim=1)  # (batch, 23)

        return self.mlp(x).squeeze(1)                   # (batch,)


n_genres = len(genre2idx)
n_langs  = len(lang2idx)
n_num    = len(NUM_COLS)

model = EmbeddingANN(n_genres, n_langs, n_num).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {total_params:,}")


## 8. Training Loop


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """
    One full pass over the data.
    optimizer=None → eval mode (no gradient updates).
    Returns (rmse, mae).
    """
    training = optimizer is not None
    model.train() if training else model.eval()

    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for num_b, genre_b, lang_b, label_b in loader:
            num_b, genre_b, lang_b, label_b = (
                num_b.to(DEVICE), genre_b.to(DEVICE),
                lang_b.to(DEVICE), label_b.to(DEVICE)
            )
            preds = model(num_b, genre_b, lang_b)
            loss  = criterion(preds, label_b)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(label_b.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    rmse = np.sqrt(mean_squared_error(all_labels, all_preds))
    mae  = mean_absolute_error(all_labels, all_preds)
    return rmse, mae


In [ ]:
NUM_EPOCHS = 20
criterion  = nn.MSELoss()
optimizer  = optim.Adam(model.parameters(), lr=0.001)

history = {'train_rmse': [], 'val_rmse': [], 'train_mae': [], 'val_mae': []}

import time
start = time.perf_counter()

for epoch in range(NUM_EPOCHS):
    tr_rmse, tr_mae = run_epoch(model, train_loader, criterion, optimizer)
    v_rmse,  v_mae  = run_epoch(model, val_loader,   criterion, None)

    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(v_rmse)
    history['train_mae'].append(tr_mae)
    history['val_mae'].append(v_mae)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS} | "
              f"Train RMSE {tr_rmse:.4f}  MAE {tr_mae:.4f} | "
              f"Val RMSE {v_rmse:.4f}  MAE {v_mae:.4f}")

ann_train_time = time.perf_counter() - start
print(f"\nTotal training time: {ann_train_time:.1f}s")


In [ ]:
# ── Learning curves ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_rmse'], label='Train'); ax1.plot(history['val_rmse'], label='Val')
ax1.set_title('RMSE over epochs'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['train_mae'],  label='Train'); ax2.plot(history['val_mae'],  label='Val')
ax2.set_title('MAE over epochs');  ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout(); plt.show()


## 9. Final Test Set Evaluation

In [ ]:
ann_tr_rmse, ann_tr_mae = run_epoch(model, train_loader, criterion, None)
ann_v_rmse,  ann_v_mae  = run_epoch(model, val_loader,   criterion, None)
ann_te_rmse, ann_te_mae = run_epoch(model, test_loader,  criterion, None)

print("EmbeddingANN Results")
print(f"  Train      RMSE: {ann_tr_rmse:.4f}   MAE: {ann_tr_mae:.4f}")
print(f"  Validation RMSE: {ann_v_rmse:.4f}   MAE: {ann_v_mae:.4f}")
print(f"  Test       RMSE: {ann_te_rmse:.4f}   MAE: {ann_te_mae:.4f}")


## 10. Save Learned Embeddings to Disk

We extract the raw weight matrices from the two `nn.Embedding` layers.
Each row in the matrix corresponds to one category index from the encoder.

Notebook 2 will load these and look up each row's genre/language vector
to use as dense features instead of OHE.


In [ ]:
# ── Extract weight matrices (detach from computation graph, move to CPU) ──
genre_weights = model.genre_emb.weight.detach().cpu().numpy()  # (n_genres, 8)
lang_weights  = model.lang_emb.weight.detach().cpu().numpy()   # (n_langs,  4)

print(f"Genre embedding matrix : {genre_weights.shape}")
print(f"Lang  embedding matrix : {lang_weights.shape}")

# ── Save weights as .npy files ──
np.save(os.path.join(EMB_DIR, 'genre_embeddings.npy'), genre_weights)
np.save(os.path.join(EMB_DIR, 'lang_embeddings.npy'),  lang_weights)

# Encoders were already saved in Step 5 — confirm all 4 files exist
for fname in ['genre_embeddings.npy', 'lang_embeddings.npy',
              'genre_encoder.json',    'lang_encoder.json']:
    fpath = os.path.join(EMB_DIR, fname)
    print(f"  {'OK' if os.path.exists(fpath) else 'MISSING'} → {fpath}")


In [ ]:
# ── Quick sanity check: look up a genre and inspect its vector ──
sample_genre = 'Drama'
idx = genre2idx.get(sample_genre, 0)
print(f"Embedding for '{sample_genre}' (index {idx}):")
print(genre_weights[idx].round(4))


## 11. Feature Importance — Integrated Gradients

Captum's Integrated Gradients requires a single tensor input, so we wrap
the model to accept one concatenated input and internally route to the right layers.
We run IG on 200 validation samples and average absolute attributions for a global view.


In [ ]:
# !pip install captum
from captum.attr import IntegratedGradients

class FlatModel(nn.Module):
    """
    Wrapper that accepts a single flat tensor for IG compatibility.
    Layout: [num(11) | lang(1) | genres(MAX_GENRES)] — all concatenated.
    The lang and genre slots carry float indices which we round back to long.
    """
    def __init__(self, emb_model, n_num, max_genres):
        super().__init__()
        self.m         = emb_model
        self.n_num     = n_num
        self.max_genres= max_genres

    def forward(self, x):
        num      = x[:, :self.n_num]
        lang_idx = x[:, self.n_num].long()
        gen_idx  = x[:, self.n_num+1:].long()
        return self.m(num, gen_idx, lang_idx).unsqueeze(1)

flat_model = FlatModel(model, len(NUM_COLS), MAX_GENRES).to(DEVICE)
ig = IntegratedGradients(flat_model)

# Build flat validation array matching the wrapper layout
val_flat = np.concatenate([
    df_val[NUM_COLS].values.astype(np.float32),
    val_lang.reshape(-1, 1).astype(np.float32),
    val_genres.astype(np.float32)
], axis=1)

N_IG = 200
all_attr = []
model.eval()
for i in range(N_IG):
    x_s = torch.tensor(val_flat[i:i+1], dtype=torch.float32).to(DEVICE)
    baseline = torch.zeros_like(x_s)
    attr, _ = ig.attribute(x_s, baselines=baseline, return_convergence_delta=True)
    all_attr.append(attr.squeeze().detach().cpu().numpy())

global_attr = np.mean(np.abs(np.array(all_attr)), axis=0)

# Build feature name list matching the flat layout
genre_col_names = [f'genre_pos_{i}' for i in range(MAX_GENRES)]
flat_feat_names = NUM_COLS + ['lang_idx'] + genre_col_names

ig_df = (pd.DataFrame({'feature': flat_feat_names, 'importance': global_attr})
         .sort_values('importance', ascending=False).head(15))

plt.figure(figsize=(8, 4))
plt.barh(ig_df['feature'][::-1], ig_df['importance'][::-1], color='steelblue')
plt.xlabel('Mean |Integrated Gradient|')
plt.title('EmbeddingANN — Top 15 Feature Attributions (n=200 val samples)')
plt.tight_layout()
plt.show()


## 12. Store ANN Results for Final Comparison

In [ ]:
# Save ANN results to a small JSON so Notebook 2 can include them in the final table
ann_results = {
    'model'         : 'EmbeddingANN',
    'train_rmse'    : round(ann_tr_rmse, 4),
    'val_rmse'      : round(ann_v_rmse,  4),
    'test_rmse'     : round(ann_te_rmse, 4),
    'train_mae'     : round(ann_tr_mae,  4),
    'val_mae'       : round(ann_v_mae,   4),
    'test_mae'      : round(ann_te_mae,  4),
    'train_time_s'  : round(ann_train_time, 1)
}
with open(os.path.join(EMB_DIR, 'ann_results.json'), 'w') as f:
    json.dump(ann_results, f, indent=2)

print("ANN results saved:")
print(json.dumps(ann_results, indent=2))
